In [82]:
import pandas as pd
import numpy as np

departments = pd.read_csv("departments.csv")
diagnosis = pd.read_csv("diagnosis.csv")
encounters = pd.read_csv("encounters.csv")
patients = pd.read_csv("patients.csv")
providers = pd.read_csv("providers.csv")
social_determinants = pd.read_csv("social_determinants.csv")
tigercensuscodes = pd.read_csv("tigercensuscodes.csv")

/var/folders/gp/m5jhv8y904l6263shvkxvlq40000gn/T/ipykernel_7057/3376211003.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  social_determinants = pd.read_csv("social_determinants.csv")


## Null Values

In [14]:
print(departments.shape[0])
departments.isnull().sum()

11597


DepartmentKey             0
Address                2884
City                   2204
County                  574
DepartmentName            0
DepartmentSpecialty     574
DepartmentType          574
PostalCode             3009
CensusTract            4575
dtype: int64

In [13]:
print(diagnosis.shape[0])
diagnosis.isnull().sum()

1531262


DiagnosisKey      0
GroupName         0
GroupCode         0
DiagnosisName     0
DiagnosisValue    0
dtype: int64

In [12]:
print(encounters.shape[0])
encounters.isnull().sum()

7675801


Date                                 0
AdmissionInstant               6627001
AdmitYear                      6627001
AdmitMonth                     6627001
AdmitDay                       6627001
AdmitHour                      6627001
AdmitMinute                    6627001
AdmissionSource                   1400
AdmissionType                     1086
DischargeInstant               6627001
DischargeYear                  6627001
DischargeMonth                 6627001
DischargeDay                   6627001
DischargeHour                  6627001
DischargeMinute                6627001
EncounterKey                         0
PatientDurableKey                    0
Type                                 0
VisitType                            0
VisitTypeDescription                 0
ProviderDurableKey                   0
AttendingProviderDurableKey          0
DischargeProviderDurableKey          0
DepartmentKey                        0
PrimaryDiagnosisKey                  0
IsEdVisit                

In [11]:
print(patients.shape[0])
patients.isnull().sum()

947685


CensusBlockGroupFipsCode         0
DurableKey                       0
FirstRace                   365398
MaritalStatus                    0
MyChartStatus                    0
OmbEthnicity                     0
OmbRace                          0
SexAssignedAtBirth               0
SexualOrientation                0
SmokingStatus                    0
VitalStatus                      0
PatientBirthYearBin           1725
dtype: int64

In [10]:
print(providers.shape[0])
providers.isnull().sum()

299075


DurableKey                0
ClinicianTitle       113067
OfficeAddress         30335
OfficeCity            30344
OfficePostalCode      30362
PrimaryDepartment         0
PrimarySpecialty      37856
Type                      0
dtype: int64

In [39]:
supporting = pd.read_csv("Social Determinant Questions and Domains.csv")
social_determinants_supported = pd.merge(social_determinants, supporting)

print(social_determinants_supported.shape[0])
social_determinants_supported.isnull().sum()

725197


DisplayName          0
AnswerText           0
EncounterKey         0
PatientDurableKey    0
Domain               0
dtype: int64

In [16]:
print(tigercensuscodes.shape[0])
tigercensuscodes.isnull().sum()

2463


GEOID              0
PopulationValue    2
CENTLAT            2
CENTLON            2
dtype: int64

## Brief Analysis

In [32]:
departments_null = departments.loc[departments["County"].isnull()]

departments_null.isnull().sum()

DepartmentKey            0
Address                574
City                   574
County                 574
DepartmentName           0
DepartmentSpecialty    574
DepartmentType         574
PostalCode             574
CensusTract            574
dtype: int64

In [51]:
patients["CensusBlockGroupFipsCode"].loc[patients["CensusBlockGroupFipsCode"] == "*Unspecified"]

1         *Unspecified
2         *Unspecified
3         *Unspecified
4         *Unspecified
5         *Unspecified
              ...     
947679    *Unspecified
947680    *Unspecified
947681    *Unspecified
947682    *Unspecified
947683    *Unspecified
Name: CensusBlockGroupFipsCode, Length: 611171, dtype: object

## Building Datasets

In [80]:
df = pd.merge(encounters, diagnosis, left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey')
df['CANCER'] = df['GroupName'].str.contains('malignant', case=False, na=False).astype(int)
cancer_df = df.loc[df["CANCER"] == 1]
cancer_df["Date"] = pd.to_datetime(cancer_df["Date"], format='mixed')

/var/folders/gp/m5jhv8y904l6263shvkxvlq40000gn/T/ipykernel_7057/3029262007.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cancer_df["Date"] = pd.to_datetime(cancer_df["Date"], format='mixed')


In [88]:
cancer_df.iloc[:, 0].head()

17   2022-02-07
18   2022-02-21
40   2022-04-12
46   2022-06-14
47   2022-05-24
Name: Date, dtype: datetime64[ns]

In [95]:
# df should be a sub-dataframe of encounters with all encounters involving the relevant diagnoses
def time_between_visits(df):
    time_btwn_avgs = []
    for patient_key in df["PatientDurableKey"].unique():
        patient_df = df.loc[df["PatientDurableKey"] == patient_key].sort_values(by = "Date", ascending = True)
        
        if len(patient_df) == 1:
            time_btwn_avgs.append(pd.to_timedelta(0, unit = 's'))
            
        else: 
            time_btwn = []
            for i in range(0, len(patient_df) - 1):
                time_btwn.append(patient_df.iloc[i+1, 0] - patient_df.iloc[i, 0])
            time_btwn_avgs.append(np.mean(time_btwn))
            
    return time_btwn_avgs

In [ ]:
cancer_waittimes = time_between_visits(cancer_df)